# TechChallenge3_Silver_to_Gold

In [3]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Inicialização do Spark no Glue
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

In [4]:
print("Iniciando processamento: Camada Silver -> Gold...")

# 2. Leitura da Camada Silver Unificada
silver_path = "s3://tech3-datahackers-datalake/silver/datahackers_unified/"
df_silver = spark.read.parquet(silver_path)

# Faz cache na memória para as agregações rodarem super rápido
df_silver.cache() 

# ==========================================
# AGREGAÇÕES DA CAMADA GOLD
# ==========================================

# Tabela 1: Panorama Salarial por Senioridade
print("Processando Tabela 1: Salários...")
df_silver.filter(F.col("current_role").isNotNull()) \
    .groupBy("year", "current_role", "seniority_level", "salary_range") \
    .agg(F.count("*").alias("total_professionals")) \
    .write.mode("overwrite").partitionBy("year") \
    .parquet("s3://tech3-datahackers-datalake/gold/salary_by_seniority/")

# Tabela 2: Diversidade (Gênero e Etnia)
print("Processando Tabela 2: Diversidade...")
df_silver.filter(F.col("gender").isNotNull()) \
    .groupBy("year", "current_role", "gender", "ethnicity") \
    .agg(F.count("*").alias("total_professionals")) \
    .write.mode("overwrite").partitionBy("year") \
    .parquet("s3://tech3-datahackers-datalake/gold/diversity_profile/")

# Tabela 3: Geografia, Modelo de Trabalho e Satisfação (Storytelling)
print("Processando Tabela 3: Geografia e Satisfação...")
df_silver.filter(F.col("region").isNotNull() & F.col("work_model").isNotNull()) \
    .groupBy("year", "region", "work_model", "job_satisfaction") \
    .agg(F.count("*").alias("total_professionals")) \
    .write.mode("overwrite").partitionBy("year") \
    .parquet("s3://tech3-datahackers-datalake/gold/geo_work_satisfaction/")

# Tabela 4: Stack Tecnológico: Nuvem, BI e Linguagem (Storytelling)
print("Processando Tabela 4: Tecnologias...")
df_silver.filter(F.col("current_role").isNotNull()) \
    .groupBy("year", "current_role", "preferred_cloud", "preferred_language", "preferred_bi") \
    .agg(F.count("*").alias("total_mentions")) \
    .write.mode("overwrite").partitionBy("year") \
    .parquet("s3://tech3-datahackers-datalake/gold/tech_stack_preferences/")

# Tabela 5: Impacto da Inteligência Artificial por Setor
print("Processando Tabela 5: Impacto IA...")
df_silver.filter(F.col("industry").isNotNull() & F.col("ai_priority").isNotNull()) \
    .groupBy("year", "industry", "ai_priority") \
    .agg(F.count("*").alias("total_companies")) \
    .write.mode("overwrite").partitionBy("year") \
    .parquet("s3://tech3-datahackers-datalake/gold/ai_adoption_impact/")

# Tabela 6: Maturidade do Mercado: Educação e Experiência (Storytelling)
print("Processando Tabela 6: Maturidade de Mercado...")
df_silver.filter(F.col("education_level").isNotNull()) \
    .groupBy("year", "current_role", "education_level", "experience_time") \
    .agg(F.count("*").alias("total_professionals")) \
    .write.mode("overwrite").partitionBy("year") \
    .parquet("s3://tech3-datahackers-datalake/gold/market_maturity/")

print("Processamento Silver -> Gold concluído com sucesso em todas as tabelas!")

Iniciando processamento: Camada Silver -> Gold...
Processando Tabela 1: Salários...
Processando Tabela 2: Diversidade...
Processando Tabela 3: Geografia e Satisfação...
Processando Tabela 4: Tecnologias...
Processando Tabela 5: Impacto IA...
Processando Tabela 6: Maturidade de Mercado...
Processamento Silver -> Gold concluído com sucesso em todas as tabelas!


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [ ]:
%idle_timeout 2880
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [ ]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='database_name', table_name='table_name')
dyf.printSchema()

#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [ ]:
df = dyf.toDF()
df.show()

#### Example: Visualize data with matplotlib


In [ ]:
import matplotlib.pyplot as plt

# Set X-axis and Y-axis values
x = [5, 2, 8, 4, 9]
y = [10, 4, 8, 5, 2]
  
# Create a bar chart 
plt.bar(x, y)
  
# Show the plot
%matplot plt

#### Example: Write the data in the DynamicFrame to a location in Amazon S3 and a table for it in the AWS Glue Data Catalog


In [ ]:
s3output = glueContext.getSink(
  path="s3://bucket_name/folder_name",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="demo", catalogTableName="populations"
)
s3output.setFormat("glueparquet")
s3output.writeFrame(DyF)